In [ ]:
!bash download_data.sh

In [7]:
df = pd.read_csv("GRCH38-cCRE.bed", sep="\t")
print(df.head())


   chr1  181251  181601  EH38E1310153  488  .  181251.1  181601.1  255,167,0  \
0  chr1  190865  191071  EH38E1310154  179  .    190865    191071  255,205,0   
1  chr1  778562  778912  EH38E1310158  759  .    778562    778912    255,0,0   
2  chr1  779086  779355  EH38E1310159  304  .    779086    779355    255,0,0   
3  chr1  779727  780060  EH38E1310160  281  .    779727    780060  255,167,0   
4  chr1  790397  790626  EH38E1310162  202  .    790397    790626  0,176,240   

        pELS,CTCF-bound       pELS  4.88403259983  enhP  E1310153  \
0       dELS,CTCF-bound       dELS       1.792822  enhD  E1310154   
1        PLS,CTCF-bound        PLS       7.598523  prom  E1310158   
2        PLS,CTCF-bound        PLS       3.046639  prom  E1310159   
3       pELS,CTCF-bound       pELS       2.817434  enhP  E1310160   
4  CTCF-only,CTCF-bound  CTCF-only       2.025385  CTCF  E1310162   

   EH38E1310153 proximal enhancer-like signature  
0    EH38E1310154 distal enhancer-like signature  
1 

In [4]:
import pandas as pd
import numpy as np

def generate_bed(input_bed, output_bed, flank=2048, seed=42):
    np.random.seed(seed)  # reproducible randomness
    
    cols = ["chrom", "start", "end", "id1", "id2", "label"]
    df = pd.read_csv(input_bed, sep="\t", header=None, names=cols)

    out_rows = []
    for _, row in df.iterrows():
        start, end = int(row["start"]), int(row["end"])
        center = (start + end) / 2

        # define central 50% of the region
        quarter = (end - start) * 0.25
        low = start + quarter
        high = end - quarter

        # pick random center inside that range
        chosen_center = np.random.randint(low, high + 1)

        # define 4096bp window
        new_start = int(chosen_center - flank)
        new_end = int(chosen_center + flank)

        out_rows.append([row["chrom"], new_start, new_end])

    out_df = pd.DataFrame(out_rows, columns=["chrom", "start", "end"])
    out_df.to_csv(output_bed, sep="\t", header=False, index=False)

# Example usage
generate_bed("GRCH38-cCRE.bed", "GRCh38_cCREs_4kb.bed")


ValueError: invalid literal for int() with base 10: 'pELS'

In [8]:
import pandas as pd
import numpy as np

def generate_bed(input_bed, output_bed, flank=2048, seed=42):
    np.random.seed(seed)
    
    # Read the file. 
    # We use usecols=[0,1,2] to get only Chrom, Start, and End.
    # We use comment='track' just in case there is a header line.
    df = pd.read_csv(input_bed, sep="\t", header=None, usecols=[0, 1, 2], 
                     names=["chrom", "start", "end"], comment='t')

    out_rows = []
    for _, row in df.iterrows():
        # Ensure they are numeric before calculations
        try:
            start, end = int(row["start"]), int(row["end"])
        except ValueError:
            # This skips lines that might be headers if "start" isn't a number
            continue
            
        # define central 50% of the region
        quarter = (end - start) * 0.25
        low = start + quarter
        high = end - quarter

        # pick random center inside that range
        # Use floor/ceil to ensure low/high are valid for randint
        chosen_center = np.random.randint(int(low), int(high) + 1)

        # define 4096bp window (2048 each side)
        new_start = int(chosen_center - flank)
        new_end = int(chosen_center + flank)

        # Basic safety: ensure start isn't negative
        if new_start < 0: new_start = 0

        out_rows.append([row["chrom"], new_start, new_end])

    out_df = pd.DataFrame(out_rows, columns=["chrom", "start", "end"])
    out_df.to_csv(output_bed, sep="\t", header=False, index=False)

# Run it
generate_bed("GRCH38-cCRE.bed", "GRCh38_cCREs_4kb.bed")

In [10]:
import os
import pandas as pd

def convert_tsv_to_vcf(folder_path):
    for filename in os.listdir(folder_path):
       
        file_path = os.path.join(folder_path, filename)

        try:
            df = pd.read_csv(file_path, sep='\t')
        except Exception as e:
            print(f"Could not read {filename}: {e}")
            continue

        required_cols = ["chrom", "pos", "strand", "ref", "alt"]
        if not all(col in df.columns for col in required_cols):
            print(f"Skipping {filename}: missing required columns.")
            continue

        df_vcf = df[required_cols].copy()

        # Convert strand from 1/-1 to +/-
        df_vcf["strand"] = df_vcf["strand"].map({1: "+", -1: "-"})
        if df_vcf["strand"].isnull().any():
            print(f"Warning: Unrecognized strand value in {filename}")

        # Rename columns
        df_vcf.columns = ["CHROM", "POS", "STRAND", "REF", "ALT"]
        df_vcf["STRAND"] = "+" # the strand info here already is for the gene, not the variant

        # Output file path
        base_name = os.path.splitext(filename)[0]
        output_path = os.path.join(folder_path, f"{base_name}.vcf")

        df_vcf.to_csv(output_path, sep='\t', index=False, header=False)
        print(f"Converted {filename} -> {base_name}.vcf")

convert_tsv_to_vcf("../data/")


Skipping UKBB_proteome.vcf: missing required columns.
Could not read caqtls.african.lcls.benchmarking.all.tsv.gz: Not a gzipped file (b'PK')
Skipping MPRA_saturation.vcf: missing required columns.
Could not read dsqtls.yoruba.lcls.benchmarking.all.tsv.gz: Not a gzipped file (b'PK')
Could not read caqtls.microglia.benchmarking.all.tsv.gz: Not a gzipped file (b'PK')
Skipping caqtls.african.lcls.benchmarking.all.README: missing required columns.
Skipping GTEx_eQTL.vcf: missing required columns.
Skipping caqtls.microglia.benchmarking.all.README: missing required columns.
Skipping SYNAPSE_METADATA_MANIFEST.tsv: missing required columns.
Could not read caqtls.smc.benchmarking.all.tsv.gz: Not a gzipped file (b'PK')
Skipping caqtls.eu.lcls.benchmarking.all.README: missing required columns.
Converted GTEx_eQTL.tsv -> GTEx_eQTL.vcf
Converted MPRA_eQTL.tsv -> MPRA_eQTL.vcf
Skipping caqtls.african.lcls.asb.benchmarking.all.README: missing required columns.
Skipping dsqtls.yoruba.lcls.benchmarking.

In [ ]:
import os
import pandas as pd

def convert_tsv_to_vcf(folder_path):
    for filename in os.listdir(folder_path):
       
        file_path = os.path.join(folder_path, filename)

        try:
            df = pd.read_csv(file_path, sep='\t')
        except Exception as e:
            print(f"Could not read {filename}: {e}")
            continue

        required_cols = ["var.chr", "var.pos_hg19", "strand", "var.allele1", "var.allele2"]
        if not all(col in df.columns for col in required_cols):
            print(f"Skipping {filename}: missing required columns.")
            continue

        df_vcf = df[required_cols].copy()

        # Convert strand from 1/-1 to +/-
        df_vcf["strand"] = df_vcf["strand"].map({1: "+", -1: "-"})
        if df_vcf["strand"].isnull().any():
            print(f"Warning: Unrecognized strand value in {filename}")

        # Rename columns
        df_vcf.columns = ["CHROM", "POS", "STRAND", "REF", "ALT"]
        df_vcf["STRAND"] = "+" # the strand info here already is for the gene, not the variant

        # Output file path
        base_name = os.path.splitext(filename)[0]
        output_path = os.path.join(folder_path, f"{base_name}.vcf")

        df_vcf.to_csv(output_path, sep='\t', index=False, header=False)
        print(f"Converted {filename} -> {base_name}.vcf")

convert_tsv_to_vcf("../data/")


In [ ]:
import os
import zipfile
import shutil

def repair_data_directory(relative_path):
    target_dir = os.path.abspath(relative_path)
    print(f"Repairing files in: {target_dir}")
    
    for filename in os.listdir(target_dir):
        file_path = os.path.join(target_dir, filename)
        
        # Skip directories
        if os.path.isdir(file_path):
            continue
            
        # Check for the ZIP header 'PK'
        try:
            with open(file_path, 'rb') as f:
                header = f.read(2)
            
            if header == b'PK':
                print(f"Detected ZIP signature in: {filename}. Extracting...")
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(target_dir)
                
                # Optional: Remove the original mislabeled file to save space
                # os.remove(file_path) 
        except Exception as e:
            print(f"Could not check {filename}: {e}")

# Run the repair first
repair_data_directory("../data/")

In [24]:
import os
import pandas as pd

def process_unzipped_tsvs(relative_path):
    target_dir = os.path.abspath(relative_path)
    
    # Define the columns once
    column_map = {
        "CHROM": ["var.chr", "chrom", "chr"],
        "POS": ["var.pos_hg38", "var.pos_hg19", "pos", "pos0"],
        "REF": ["var.allele1", "var.ref", "ref", "var.POSTallele"],
        "ALT": ["var.allele2", "var.alt", "alt", "var.ALTallele"]
    }

    for filename in os.listdir(target_dir):
        # 1. ONLY process .tsv files
        # 2. IGNORE the VCFs we are creating (to avoid an infinite loop or reading our own output)
        if filename.endswith(".tsv") and not filename.endswith(".vcf"):
            
            file_path = os.path.join(target_dir, filename)
            print(f"Reading: {filename}")

            try:
                # No need for compression='infer' if they are already unzipped
                df = pd.read_csv(file_path, sep='\t', low_memory=False)
                
                # Logic to find columns...
                found = {k: next((s for s in syns if s in df.columns), None) for k, syns in column_map.items()}
                
                if None in found.values():
                    print(f"  ! Missing columns in {filename}. Skipping.")
                    continue

                # Create VCF
                vcf_df = pd.DataFrame({
                    "CHROM": df[found["CHROM"]],
                    "POS": df[found["POS"]],
                    "ID": ".",
                    "REF": df[found["REF"]],
                    "ALT": df[found["ALT"]],
                    "QUAL": ".",
                    "FILTER": "PASS",
                    "INFO": "STRAND=+"
                })

                output_name = filename.replace(".tsv", "") + ".vcf"
                vcf_df.to_csv(os.path.join(target_dir, output_name), sep='\t', index=False, header=False)
                print(f"  ✔ Saved: {output_name}")

            except Exception as e:
                print(f"  ✘ Error reading {filename}: {e}")

process_unzipped_tsvs("../data/")

Reading: SYNAPSE_METADATA_MANIFEST.tsv
  ! Missing columns in SYNAPSE_METADATA_MANIFEST.tsv. Skipping.
Reading: GTEx_eQTL.tsv
  ✔ Saved: GTEx_eQTL.vcf
Reading: MPRA_eQTL.tsv
  ✔ Saved: MPRA_eQTL.vcf
Reading: CAGI5_saturation.tsv
  ✔ Saved: CAGI5_saturation.vcf
Reading: dsqtls.yoruba.lcls.benchmarking.all.tsv
  ✔ Saved: dsqtls.yoruba.lcls.benchmarking.all.vcf
Reading: bqtls.pu1.lcls.benchmarking.all.tsv
  ✔ Saved: bqtls.pu1.lcls.benchmarking.all.vcf
Reading: GTEx_outlier.tsv
  ✔ Saved: GTEx_outlier.vcf
Reading: caqtls.african.lcls.asb.benchmarking.all.tsv
  ! Missing columns in caqtls.african.lcls.asb.benchmarking.all.tsv. Skipping.
Reading: MPRA_saturation.tsv
  ✔ Saved: MPRA_saturation.vcf
Reading: UKBB_proteome.tsv
  ✔ Saved: UKBB_proteome.vcf
Reading: caqtls.microglia.benchmarking.all.tsv
  ✔ Saved: caqtls.microglia.benchmarking.all.vcf
Reading: GEL_RNA.tsv
  ✔ Saved: GEL_RNA.vcf
Reading: finetune_gtex.tsv
  ✔ Saved: finetune_gtex.vcf
Reading: caqtls.african.lcls.benchmarking.all.ts

In [13]:
df = pd.read_csv("../data/dsqtls.yoruba.lcls.benchmarking.all.vcf", sep='\t', low_memory=False)

In [14]:
df_vcf = df.iloc[:, [3, 4]]

In [22]:
print(df[df.iloc[:, 5] == 'S'])

Empty DataFrame
Columns: [chr1, 856583, ., Unnamed: 3, AG, ..1, PASS, STRAND=+]
Index: []
